# Compresr × LlamaIndex

Three groups of integrations for LlamaIndex, each demonstrated with a **without / with** compression comparison on a real Apple 10-K filing.

| Group | Symbol | What it does |
|---|---|---|
| **A. Postprocessor** | `CompresrNodePostprocessor` | `BaseNodePostprocessor` — drop-in for `LongLLMLinguaPostprocessor`. |
| **B. Tool wrapper** | `wrap_tool_with_compresr` | Wraps any `FunctionTool` so the return value is compressed. |
| **C. Memory** | `CompresrMemoryBlock` | `BaseMemoryBlock` — token-level history truncation, no LLM call. |

Customer scenario: a financial analyst asks narrow questions over Apple's FY2024 10-K. Top-k retrieval pulls back ~10k tokens of dense filings prose per query — perfect setup for query-aware compression of retrieved chunks.

## Install

In [1]:
%pip install -q -e "..[llamaindex]" python-dotenv requests openai tiktoken llama-index-llms-openai llama-index-embeddings-openai


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Setup

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        break

assert os.environ.get("COMPRESR_API_KEY"), "COMPRESR_API_KEY not set"
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not set"
print("API keys loaded.")

API keys loaded.


## Shared setup — Apple 10-K filing + savings helper

Every demo loads Apple's FY2024 Form 10-K directly from SEC EDGAR. The `print_savings` helper builds the same $/month table across groups.

In [3]:
import re
import requests

FILING_URL = "https://www.sec.gov/Archives/edgar/data/320193/000032019324000123/aapl-20240928.htm"
FILING_TITLE = "Apple Inc. Form 10-K (fiscal year ended September 28, 2024)"
QUERY = "What were Apple total net sales and Services revenue for fiscal year 2024?"

HEADERS = {"User-Agent": "compresr-sdk-tutorial compresr.founders@gmail.com"}

def fetch_sec_filing(url: str) -> str:
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    html = r.text
    html = re.sub(r"<script[\s\S]*?</script>", " ", html, flags=re.I)
    html = re.sub(r"<style[\s\S]*?</style>", " ", html, flags=re.I)
    html = re.sub(r"<[^>]+>", " ", html)
    html = re.sub(r"&nbsp;|&#160;", " ", html)
    html = re.sub(r"&amp;", "&", html)
    html = re.sub(r"&[a-zA-Z#0-9]+;", " ", html)
    html = re.sub(r"[ \t]+", " ", html)
    html = re.sub(r"\n\s*\n+", "\n\n", html)
    return html.strip()

filing_text = fetch_sec_filing(FILING_URL)
print(f"Filing: {FILING_TITLE}")
print(f"Length: {len(filing_text):,} chars (~{len(filing_text) // 4:,} tokens)")

CALLS_PER_DAY = 1_000
DAYS_PER_MONTH = 30

def monthly_cost(tokens: int, price_per_1m: float) -> float:
    return tokens * CALLS_PER_DAY * DAYS_PER_MONTH * price_per_1m / 1_000_000

def print_savings(raw_tokens: int, cmp_tokens: int) -> None:
    prices = {"gpt-4o-mini": 0.15, "gpt-4o": 2.50, "claude-sonnet": 3.00}
    print(f"{'Model':<16}{'Raw $/mo':>14}{'Compresr $/mo':>17}{'Saved $/mo':>14}")
    for name, price in prices.items():
        r = monthly_cost(raw_tokens, price)
        c = monthly_cost(cmp_tokens, price)
        print(f"{name:<16}{r:>14,.2f}{c:>17,.2f}{r - c:>14,.2f}")

Filing: Apple Inc. Form 10-K (fiscal year ended September 28, 2024)
Length: 218,136 chars (~54,534 tokens)


## A. `CompresrNodePostprocessor` — RAG postprocessing

**What it does:** `BaseNodePostprocessor` that batch-compresses every retrieved `NodeWithScore` before it reaches the LLM.

**When to use it:** any RAG pipeline. Drop-in for `LongLLMLinguaPostprocessor` — no local GPU required, hosted batch endpoint compresses all retrieved nodes in one call. Wire it through `index.as_query_engine(node_postprocessors=[...])` and the rest of the pipeline is unchanged.

Below: build a `VectorStoreIndex` over the 10-K, retrieve top-k chunks for the analyst question, then call **`gpt-4o-mini`** twice — once on the raw retrieved context, once on the Compresr-compressed context — to confirm answers stay equivalent while input tokens (and $ cost) drop.

In [4]:
import tiktoken
from openai import OpenAI
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import QueryBundle
from llama_index.embeddings.openai import OpenAIEmbedding
from compresr.integrations.llamaindex import CompresrNodePostprocessor

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.llm = None
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=64)

doc = Document(text=filing_text, metadata={"title": FILING_TITLE})
index = VectorStoreIndex.from_documents([doc])
retriever = index.as_retriever(similarity_top_k=8)
retrieved = retriever.retrieve(QueryBundle(query_str=QUERY))

pp = CompresrNodePostprocessor(
    api_key=os.environ["COMPRESR_API_KEY"],
    compression_model="latte_v1",
    target_compression_ratio=0.5,
    min_tokens=100,
)
compressed_nodes = pp.postprocess_nodes(retrieved, query_bundle=QueryBundle(query_str=QUERY))

enc = tiktoken.encoding_for_model("gpt-4o-mini")
raw_context = "\n\n".join(n.node.get_content() for n in retrieved)
cmp_context = "\n\n".join(n.node.get_content() for n in compressed_nodes)
raw_ctx_tokens = len(enc.encode(raw_context))
cmp_ctx_tokens = len(enc.encode(cmp_context))

client = OpenAI()
PROMPT = (
    "You are a financial analyst. Use ONLY the provided 10-K excerpts to answer. "
    "Be concise and cite specific dollar figures when present.\n\n"
    "Question: {q}\n\nExcerpts:\n{ctx}\n\nAnswer:"
)

def ask(ctx: str):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": PROMPT.format(q=QUERY, ctx=ctx)}],
        temperature=0,
    )
    return resp.choices[0].message.content, resp.usage.prompt_tokens, resp.usage.completion_tokens

raw_answer, raw_in, raw_out = ask(raw_context)
cmp_answer, cmp_in, cmp_out = ask(cmp_context)

print(f"Retrieved {len(retrieved)} chunks for: {QUERY!r}")
print()
print(f"Context tokens (tiktoken):   raw={raw_ctx_tokens:,}  compresr={cmp_ctx_tokens:,}  "
      f"({(1 - cmp_ctx_tokens / raw_ctx_tokens) * 100:.1f}% smaller)")
print(f"gpt-4o-mini input tokens:    raw={raw_in:,}  compresr={cmp_in:,}  "
      f"({(1 - cmp_in / raw_in) * 100:.1f}% smaller)")
saved_per_1k = (raw_in - cmp_in) * 1_000 * 0.15 / 1_000_000
print(f"Saved per 1,000 requests @ $0.15/M input tokens: ${saved_per_1k:,.2f}")
print()
print("--- Answer with RAW retrieved context ---")
print(raw_answer)
print()
print("--- Answer with COMPRESR-compressed context ---")
print(cmp_answer)
print()
print_savings(raw_in, cmp_in)
from IPython.display import display
from _demo_utils import compresr_diff_html
print("\nWord-level diff of retrieved context (showing first ~20k chars):")
display(compresr_diff_html(raw_context, cmp_context))

LLM is explicitly disabled. Using MockLLM.


Retrieved 8 chunks for: 'What were Apple total net sales and Services revenue for fiscal year 2024?'

Context tokens (tiktoken):   raw=3,338  compresr=1,941  (41.9% smaller)
gpt-4o-mini input tokens:    raw=3,395  compresr=1,998  (41.1% smaller)
Saved per 1,000 requests @ $0.15/M input tokens: $0.21

--- Answer with RAW retrieved context ---
For fiscal year 2024, Apple reported total net sales of **$391,035 million** and Services revenue of **$96,169 million**.

--- Answer with COMPRESR-compressed context ---
For fiscal year 2024, Apple reported total net sales of **$391,035 million** and Services revenue of **$96,169 million**.

Model                 Raw $/mo    Compresr $/mo    Saved $/mo
gpt-4o-mini              15.28             8.99          6.29
gpt-4o                  254.62           149.85        104.78
claude-sonnet           305.55           179.82        125.73

Word-level diff of retrieved context (showing first ~20k chars):


## B. `wrap_tool_with_compresr` — tool output compression

**What it does:** wraps any `FunctionTool` so the return value is compressed transparently before reaching the agent.

**When to use it:** LlamaIndex agents (`AgentRunner`, `ReActAgent`, workflows) where tools return long content. Zero-rewire — wrap the tool, register the wrapped version, done.

Below: a `sec_filing_lookup` tool that fetches a 10-K from SEC EDGAR. Without the wrapper an agent burns ~80k input tokens digesting the raw filing on every call; with the wrapper the same call returns only the query-relevant portion.

In [5]:
from llama_index.core.tools import FunctionTool
from compresr.integrations.llamaindex import wrap_tool_with_compresr

def sec_filing_lookup(query: str, url: str = FILING_URL) -> str:
    return fetch_sec_filing(url)

filing_tool = FunctionTool.from_defaults(
    fn=sec_filing_lookup,
    name="sec_filing_lookup",
    description="Fetch the full text of a SEC 10-K filing. Pass the analyst question as `query`.",
)

wrapped = wrap_tool_with_compresr(
    filing_tool,
    api_key=os.environ["COMPRESR_API_KEY"],
    compression_model="latte_v1",
    query_arg="query",
    target_compression_ratio=0.3,
    min_tokens=500,
)

TOOL_QUERY = "What were Apple total operating expenses and R&D spend in fiscal 2024?"
raw_out = filing_tool.call(query=TOOL_QUERY).raw_output
cmp_out = wrapped.call(query=TOOL_QUERY).raw_output

raw_t = len(enc.encode(raw_out))
cmp_t = len(enc.encode(cmp_out))
print(f"Tool query: {TOOL_QUERY!r}")
print(f"Without wrapper: {raw_t:>7,} tokens of filing text reach the agent")
print(f"With wrapper:    {cmp_t:>7,} tokens reach the agent   "
      f"({(1 - cmp_t / raw_t) * 100:.1f}% smaller)")

Tool query: 'What were Apple total operating expenses and R&D spend in fiscal 2024?'
Without wrapper:  48,195 tokens of filing text reach the agent
With wrapper:     35,587 tokens reach the agent   (26.2% smaller)


## C. `CompresrMemoryBlock` — long-term agent memory

**What it does:** `BaseMemoryBlock` that aggregates messages into a buffer and compresses on `atruncate` — instead of dropping or LLM-summarizing.

**When to use it:** long-running agents that flush short-term history into long-term blocks. Built-in summary memory burns an LLM call on every truncation; this uses token-level compression — faster, cheaper, preserves original wording of the surviving tokens.

Below: a multi-turn analyst session reviewing the 10-K. The block accumulates Q&A turns containing long filing excerpts and compresses the buffer when the token budget is exceeded.

In [6]:
from compresr.integrations.llamaindex import CompresrMemoryBlock

conversation = "\n\n".join([
    f"user: {QUERY}",
    "assistant: Apple reported total net sales of $391.0B for fiscal 2024, with Services revenue of $96.2B.",
    "user: How did Products revenue split between iPhone, Mac, iPad and Wearables?",
    f"assistant: From the 10-K Products segment table:\n{filing_text[:4000]}",
    "user: What were operating expenses and R&D in fiscal 2024?",
    f"assistant: From the Consolidated Statements of Operations:\n{filing_text[4000:9000]}",
    "user: What share repurchase activity did Apple report?",
    f"assistant: From the capital return discussion:\n{filing_text[9000:13000]}",
])

without_tokens = len(enc.encode(conversation))

block = CompresrMemoryBlock(
    api_key=os.environ["COMPRESR_API_KEY"],
    target_token=1_500,
    compression_model="latte_v1",
    query=QUERY,
)
with_buffer = await block.atruncate(conversation, tokens_to_truncate=max(1, without_tokens - 1_500))
with_tokens = len(enc.encode(with_buffer or conversation))

print(f"Without memory block: {without_tokens:>7,} tokens kept (raw conversation history)")
print(f"With memory block:    {with_tokens:>7,} tokens kept   "
      f"({(1 - with_tokens / without_tokens) * 100:.1f}% smaller)")
print()
print("Wire into LlamaIndex Memory:")
print("  from llama_index.core.memory import Memory")
print("  memory = Memory.from_defaults(token_limit=8_000, memory_blocks=[block])")

Without memory block:   4,976 tokens kept (raw conversation history)
With memory block:      1,612 tokens kept   (67.6% smaller)

Wire into LlamaIndex Memory:
  from llama_index.core.memory import Memory
  memory = Memory.from_defaults(token_limit=8_000, memory_blocks=[block])


## How they compose

Different layers for different bottlenecks:

- **RAG retrieval:** `CompresrNodePostprocessor` between the retriever and the query engine.
- **Tool calls:** `wrap_tool_with_compresr` for any noisy `FunctionTool`.
- **Conversation history:** `CompresrMemoryBlock` for long-running agents.

All three use Compresr's `latte_v2` model — query-aware, token-level compression with no LLM round-trip.